In [1]:
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import Input, LSTM, Dense, Embedding, Dot, Activation, Concatenate
from tensorflow.keras.models import Model
import numpy as np

In [2]:
english_sentences = [
    "i like coffee",
    "i drink milk",
    "you love books",
    "you read newspapers",
    "she likes flowers",
    "she cooks dinner",
    "he plays cricket",
    "he watches tv",
    "we go to college",
    "we study together",
    "they eat fruits",
    "they drink juice",
    "i am very happy",
    "you are a teacher",
    "she is a student",
    "he is my brother",
    "we love our country",
    "they help people",
    "i write a letter",
    "you open the door",
    "she sings a song",
    "he drives a car",
    "we clean the house",
    "they play games"
]

tamil_sentences = [
    "<start> எனக்கு காபி பிடிக்கும் <end>",
    "<start> நான் பால் குடிக்கிறேன் <end>",
    "<start> நீங்கள் புத்தகங்களை நேசிக்கிறீர்கள் <end>",
    "<start> நீங்கள் செய்தித்தாள்களை படிக்கிறீர்கள் <end>",
    "<start> அவளுக்கு மலர்கள் பிடிக்கும் <end>",
    "<start> அவள் இரவு உணவு சமைக்கிறாள் <end>",
    "<start> அவன் கிரிக்கெட் விளையாடுகிறான் <end>",
    "<start> அவன் தொலைக்காட்சி பார்க்கிறான் <end>",
    "<start> நாங்கள் கல்லூரிக்கு செல்கிறோம் <end>",
    "<start> நாங்கள் ஒன்றாக படிக்கிறோம் <end>",
    "<start> அவர்கள் பழங்கள் சாப்பிடுகிறார்கள் <end>",
    "<start> அவர்கள் ஜூஸ் குடிக்கிறார்கள் <end>",
    "<start> நான் மிகவும் மகிழ்ச்சியாக இருக்கிறேன் <end>",
    "<start> நீங்கள் ஒரு ஆசிரியர் <end>",
    "<start> அவள் ஒரு மாணவி <end>",
    "<start> அவன் என் சகோதரன் <end>",
    "<start> நாங்கள் எங்கள் நாட்டை நேசிக்கிறோம் <end>",
    "<start> அவர்கள் மக்களுக்கு உதவுகிறார்கள் <end>",
    "<start> நான் ஒரு கடிதம் எழுதுகிறேன் <end>",
    "<start> நீங்கள் கதவை திறக்கிறீர்கள் <end>",
    "<start> அவள் ஒரு பாடல் பாடுகிறாள் <end>",
    "<start> அவன் ஒரு கார் ஓட்டுகிறான் <end>",
    "<start> நாங்கள் வீட்டை சுத்தம் செய்கிறோம் <end>",
    "<start> அவர்கள் விளையாட்டுகள் விளையாடுகிறார்கள் <end>"
]

In [3]:
src_tokenizer = Tokenizer(oov_token="<OOV>")
src_tokenizer.fit_on_texts(english_sentences)
src_seq = src_tokenizer.texts_to_sequences(english_sentences)

tgt_tokenizer = Tokenizer(filters='', oov_token="<OOV>")
tgt_tokenizer.fit_on_texts(tamil_sentences)
tgt_seq = tgt_tokenizer.texts_to_sequences(tamil_sentences)

src_vocab_size = len(src_tokenizer.word_index) + 1
tgt_vocab_size = len(tgt_tokenizer.word_index) + 1

In [4]:
max_src_len = max(len(s) for s in src_seq)
max_tgt_len = max(len(s) for s in tgt_seq)

encoder_input = pad_sequences(src_seq, maxlen=max_src_len, padding='post')

decoder_input = pad_sequences(
    [s[:-1] for s in tgt_seq],
    maxlen=max_tgt_len - 1,
    padding='post'
)

decoder_target = pad_sequences(
    [s[1:] for s in tgt_seq],
    maxlen=max_tgt_len - 1,
    padding='post'
)

In [5]:
embedding_dim = 32
latent_dim = 64

In [6]:
encoder_inputs = Input(shape=(max_src_len,))
encoder_embedding = Embedding(src_vocab_size, embedding_dim)(encoder_inputs)

encoder_lstm = LSTM(latent_dim, return_sequences=True, return_state=True)
encoder_outputs, state_h, state_c = encoder_lstm(encoder_embedding)
encoder_states = [state_h, state_c]

In [7]:
decoder_inputs = Input(shape=(max_tgt_len - 1,))
decoder_embedding = Embedding(tgt_vocab_size, embedding_dim)(decoder_inputs)

decoder_lstm = LSTM(latent_dim, return_sequences=True, return_state=True)
decoder_outputs, _, _ = decoder_lstm(
    decoder_embedding,
    initial_state=encoder_states
)

In [8]:
attention = Dot(axes=[2, 2])([decoder_outputs, encoder_outputs])
attention = Activation('softmax')(attention)

context = Dot(axes=[2, 1])([attention, encoder_outputs])
decoder_combined = Concatenate(axis=-1)([context, decoder_outputs])

In [9]:
output = Dense(tgt_vocab_size, activation='softmax')(decoder_combined)

In [10]:
model = Model([encoder_inputs, decoder_inputs], output)

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 4)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_1       │ (None, 5)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 4, 32)     │      1,888 │ input_layer[0][0] │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_1         │ (None, 5, 32)     │      1,984 │ input_layer_1[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm (LSTM)         │ [(None, 4, 64),   │     24,832 │ embedding[0][0]   │
│                     │ (None, 64),       │            │                   │
│                     │ (None, 64)]       │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_1 (LSTM)       │ [(None, 5, 64),   │     24,832 │ embedding_1[0][0… │
│                     │ (None, 64),       │            │ lstm[0][1],       │
│                     │ (None, 64)]       │            │ lstm[0][2]        │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dot (Dot)           │ (None, 5, 4)      │          0 │ lstm_1[0][0],     │
│                     │                   │            │ lstm[0][0]        │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation          │ (None, 5, 4)      │          0 │ dot[0][0]         │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dot_1 (Dot)         │ (None, 5, 64)     │          0 │ activation[0][0], │
│                     │                   │            │ lstm[0][0]        │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 5, 128)    │          0 │ dot_1[0][0],      │
│ (Concatenate)       │                   │            │ lstm_1[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 5, 62)     │      7,998 │ concatenate[0][0] │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 61,534 (240.37 KB)

 Trainable params: 61,534 (240.37 KB)

 Non-trainable params: 0 (0.00 B)

In [11]:
model.fit(
    [encoder_input, decoder_input],
    np.expand_dims(decoder_target, -1),
    epochs=300,
    batch_size=2,
    shuffle=False,
    verbose=1
)

Epoch 1/300
12/12 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - accuracy: 0.1248 - loss: 4.1214
Epoch 2/300
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.3257 - loss: 4.0813
Epoch 3/300
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.3594 - loss: 4.0005
Epoch 4/300
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.3431 - loss: 3.6398
Epoch 5/300
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.2168 - loss: 3.0772
Epoch 6/300
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.3120 - loss: 2.9963
Epoch 7/300
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.3698 - loss: 2.9485
Epoch 8/300
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.3698 - loss: 2.8920
Epoch 9/300
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.3698 - loss: 2.8393
Epoch 10/300
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.3782 - loss: 2.7822
Epoch 11/300
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.4118 - loss: 2.7202
Epoch 12/300
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy:

In [12]:
reverse_target_index = {v: k for k, v in tgt_tokenizer.word_index.items()}

def translate(sentence):
    seq = src_tokenizer.texts_to_sequences([sentence])
    seq = pad_sequences(seq, maxlen=max_src_len, padding='post')

    target_seq = np.zeros((1, max_tgt_len - 1))
    target_seq[0, 0] = tgt_tokenizer.word_index['<start>']

    decoded_sentence = []

    for i in range(max_tgt_len - 1):
        predictions = model.predict([seq, target_seq], verbose=0)
        predicted_id = np.argmax(predictions[0, i])
        word = reverse_target_index.get(predicted_id, '')

        if word == '<end>' or word == '':
            break

        decoded_sentence.append(word)
        target_seq[0, i + 1] = predicted_id

    return ' '.join(decoded_sentence)

In [13]:
print("English :", "i like coffee")
print("Tamil   :", translate("i like coffee"))

print("English :", "we go to college")
print("Tamil   :", translate("we go to college"))

print("English :", "she sings a song")
print("Tamil   :", translate("she sings a song"))

English : i like coffee
Tamil   : எனக்கு காபி பிடிக்கும்
English : we go to college
Tamil   : நாங்கள் கல்லூரிக்கு செல்கிறோம்
English : she sings a song
Tamil   : அவள் ஒரு பாடல் பாடுகிறாள்
